In [1]:
import psycopg2
from kafka import KafkaProducer
import json
import time

In [2]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="streaming_db",
    user="postgres",
    password=""
)

In [3]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

producer.send("crypto_stream", {"coin": "BTC", "price": 30000})
producer.flush()
print("✅ berhasil kirim")

✅ berhasil kirim


In [4]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

cur = conn.cursor()
last_id = 0

while True:
    cur.execute(
        "SELECT * FROM crypto_price WHERE id > %s ORDER BY id",
        (last_id,)
    )
    rows = cur.fetchall()

    for r in rows:
        last_id = r[0]
        producer.send("crypto_stream", {
            "coin": r[1],
            "price": r[2],
            "time": str(r[3])
        })

    producer.flush()
    time.sleep(3)

KeyboardInterrupt: 